# Report - Master Thesis Project

## Motivation
The motivation for this thesis stems from the increasing relevance of Quantum Machine Learning (QML) and its potential to significantly shape future computing paradigms.
Encouraged by the provision of quantum resources through IBM, this project aims to pursue a technologically feasible approach by practically exploring the innovative concept of Quantum Reservoir Computing (QRC).
As existing literature already reports the successful implementation of reservoir structures in various applications[1][2], this work investigates how this novel, physically-inspired approach compares to the classical, statistically-grounded ARIMA model.
This endeavor combines the ambition to explore new technical frontiers with a rigorous performance analysis based on currently available quantum technology.

## Hardware
All calculations and simulations were made on my home computer, a Framework 13 with the following specifications:

| Komponente | Spezifikation |
|---|---|
| Prozessor (CPU) | AMD Ryzen 7 7840HS (8 Kerne, 16 Threads) |
| Arbeitsspeicher (RAM) | 64 GB DDR5 |
| Festplatte (SSD) | 1 TB NVMe |

## Software Environment

### List of most critical pieces of software

| Category | Specification                              |
|---|--------------------------------------------|
| Operating System | Fedora 41 (Linux)                          |
| Integrated Development Environment | PyCharm 2024.3.5 (Professional Edition)    |
| Runtime Environment | Python 3.11 (Conda-based distribution)     |
| Quantum Computing Stack | Qiskit 2.3.1, Qiskit-Aer 0.17.2            |
| Quantum Reservoir Frameworks | qreservoir 0.5.0, qulacs 0.6.13            |
| Statistical Modeling (ARIMA) | pmdarima 2.1.1, statsmodels, Prophet 1.3.0 |
| Data Analysis & Processing | NumPy 2.3.1, Pandas 2.3.0, SciPy           |
| Visualization | Matplotlib 3.10.7 |

### Additional remarks

   1. The fact that ```Qiskit``` and ```Qiskit-Aer``` is present alongside ```qreservoir``` and ```qulacs``` reflects the conflict between prototyping and real implementation.
      1. ```qreservoir``` was used to find feasible quantum reservoirs in prototyping sessions. It is built on the ```qulacs``` framework.
      2. In future, ```Qiskit``` may be the framework of choice when an implementation of a real-life quantum reservoir is required.
   2. Due to the short time period of working on this project, a few technical mistakes/omissions were made:
      1. No random seed was used. Therefore, I exported the entire ```jupyter``` notebook in which the calculations were made.




## Technology
### Classical time-series models
Classical time series analysis is done with parametric models.
One class of models are $ARIMA(p,d,q)$ models.
ARIMA models are combinations of $AR$ ("autoregressive") models, $MA$ ("moving average") models and model differences ("integrated").
Each has a set of parameters whose respective counts are denoted by the aforementioned $(p,d,q)$.
We will not go into details concerning the realized parameters, but restrict ourselves to the fact that the restriction on the model size was chosen to be $(4,2,4)$.
This means, that the set of $AR$-related parameters has a count of at most $4$ coefficients, etc. [7]

### Quantum Reservoir Computing
Quantum Reservoir Computing (QRC) is a variant of the more general concept of Reservoir Computing (RC).
Reservoir Computing (RC) explores alternative physical media for computation—ranging from biological systems to a literal bucket of water—leveraging their inherent non-linear dynamics as a high-dimensional computational resource to solve complex problems.
Not everything can be used as a reservoir.

Quantum systems, as indicated in [1] and [2], can be used as a reservoir.
Having identified the platform on which RC shall be performed, the task is to encode the input so that the dynamics can work their magic, and to train an output layer for
the interpretation of the dynamics reaction.
With quantum systems this amounts to:
   1. Encoding the input to a quantum system.
   2. Choosing the specifics of the system, such as the amount of qubits, the relation between qubits, and the information one would like to read out ("measurement") from the system.
   3. Training a linear regression for interpretation of the measurements.[4][5][6]
These steps are what is the central part of my implementation and what will be the information from which to draw the conclusions in my work.

## Data
The data on which I did my analysis were retrieved from Geosphere Austria.
Geosphere Austria is a service to the Austrian government and public with respect to meteorological and climatological matters.
Geosphere Austria provides an API to the meteorological data collected by stations in Austria.

In my current work I focused on the data collected by station $107$ ("Wien Unterlaa").
The data were collected every 10 minutes from 1 January 1992 until 31 December 2025.
The variable of interest is the air temperature measured 2 meters above the ground ("```tl```").
Due to the temperatures nature, the data should be quite smooth and rather well-behaved, i.e., not big jumps are to be expected between two adjacent measurements.


## Methodology

An $ARIMA(p,d,q)$ model was selected using the Auto-ARIMA framework ```pmdarima```.
```pmdarima``` searched the optimal model by ```stepwise``` fitting of models, and returning the model with the lowest score of the chosen information criterion (IC).
In this analysis, I chose the "Akaike information criterion" which is the default IC of the function ```auto_arima()```.
```stepwise``` denotes the strategy with which the next model is chosen in the search.
It denotes an algorithm that was devised by Hyndman and Khandakar in their 2008 article "Automatic Time Series Forecasting: The forecast Package for R"[3].

The optimal $ARIMA$ model and the resulting one-step-predictions (10 minutes) are used as a benchmark against which the QRC models are compared.


## Findings
### Classical time series analysis
I used column "```tl```" for the last 3 years of data that were downloaded (entire years 2023, 2024, 2025) as the entire data set.
Thereof, $1000$ days were used as training data.
On that data set, the optimal model was chosen to be $ARIMA(4,2,4)$.

As can be seen in the in-sample mean absolute error and the in-sample standard deviation, the model was quite well fit. The one-step prediction deviates from the realised value by $0.18$ degrees Celsius on average, and with a standard deviation of $0.28$ degrees.
Indeed, the fit seems to be a very good one: The one-step prediction performance of the ARIMA model is even better. The mean absolute error is lower than in the training evaluation: $0.2$ in the tests compared with $0.29$ in the training data.
And also the standard deviation is smaller with $0.32$ instead of $0.43$.
In conclusion, the ARIMA model is a very competitive model to be compared against.

The implementation can be found in the notebook ```notebooks/ts-classical.ipynb``` or together with the evaluated functions in ```notebooks/ts-classical.html```.

### QRC
I again used column "```tl```". This time it is only for the last $100000$ measurements, which roughly amounts to 2 years worth of data.
It has to be said that the overall prediction quality is very poor.
No major trend could be captured by any model that was trained.
Most of the models showed a constant prediction value over all measurements.
In some cases a little more variation in the prediction data can be observed although with no discernible pattern.

Let's have a look at the configurations in more detail.
The implementation can be found in the notebook ```notebooks/QRC-simulation.ipynb``` or together with the evaluated functions in ```notebooks/QRC-simulation.html```.

#### Base model
This was the first attempt of training a QRC model.
The goal was to have a running fitting/prediction pipeline.
All the parameters were set to ```1```.
As the encoder I chose the ```CHEEncoder```.
The ```CHEEncoder``` "[i]mplements a reuploading stretegy [sic!] using CZ and random pauli rotation gates with a cyclic entangling structure." [8]
In so far, the model is successful.
However, the prediction values are constant and, hence, the model is not very useful.

#### Vary the number of ancilla qubits and observables
My next attempt aimed at the capacity of the reservoir.
The ancilla qubits and the observables from 1 qubit were increased in parallel.
All other parameters remaining equal, I thought that an increased number of qubits within the quantum system and an increased number of observables from the system might provide a better predictive power.
This was proven to be wrong, as can be seen in the models' outputs, for all the numbers of qubits and observables.
Again all the prediction values were constant.
Suffice to say, the mean absolute errors and standard deviations were large.
Sequentially increasing the number of qubits didn't change the situation.
Probably the missing entanglement between the qubits led to too little internal dynamics within the system for it to have predictive capacity.

#### Variation in ancilla qubits, and make the observables more involved
As indicated in the paragraph above, the missing entanglement between the observed qubits might not have been able to produce enough predictive power.
That is why, as the next step, I tried to increase the complexity and added entangling gates.
The inspiration was drawn from the example on qreservoir's docu page: https://owenagnel.github.io/qreservoir/qreservoir.html#example
I made a small variation so that not the first three observable but every third observable gets the respective treatment.
In particular, the treatment applied to the observables is the application of a Pauli-operator (```X```, ```Y```, ```Z```) to the $i$-th qubit within the observable.

This approach also didn't make the difference.
Again, a constant prediction value. Sequentially increasing the number of qubits didn't change the situation.

#### Vary the depth of the model
In the next step, I tried varying the depth of the reservoir.
As can be seen in the plots, again, there is a constant prediciton value for all measurements. Sequentially increasing the depth didn't change the situation.

#### Trying out another reservoir - RotationReservoir
Since varying the numeric parametrisation didn't yield any exploitable results, I tried changing the reservoir class to RotationReservoir.
"The RandomReservoir [!] class simulates a reservoir with random dynamics. The reservoir is made up of random rotations and CZ gates in a cyclic entangling structure."

With this approach I was able to at least produce non-constant results.
However, as can be seen from the mean absolute errors and their standard deviations, this model is also not very useful.
It has to be stated that the simulation on my home computer with 8 ancillary qubits had to be interrupted after one hour.

#### ExpEncoder
My last attempt was to change the encoder, with the reservoir remaining fixed at RotationReservoir.

Sequentially increasing the number of qubits didn't change the situation.

## Conclusion
I've been warned by several articles and books that quantum reservoirs are very hard to train for somebody that has no experience in quantum (reservoir) computing.
This is certainly true for the project at hand.
It is not clear which are the relevant parameters and what is the effect of changing one.
Hence, it is does not come as a surprise that no usable models were produced in the course of this master thesis project.


## Outlook
I will continue my search for a reservoir model that captures some of the patterns shown by the original time series and successfully captured by the ARIMA model.
Furthermore, I need to gain a deeper understanding of quantum computing in general and quantum reservoir computing in particular in order for me to develop an intuition of how the QRC models may be trained more efficiently.
Also, for the sake of closure it would be nice to develop an implementation that runs on IBM's quantum computers.
This would be needed in order to further assess the applicability of quantum reservoir computing apart from simulations and theoretical reasoning.

## Literature
[1] Otieno2026 - A Quantum Reservoir Computing Approach to Quantum Stock Price Forecasting in Quantum-Invested Markets

[2] Domingo2024 - Optimal quantum reservoir computing for market forecasting: An application to fight food price crises

[3] Hyndman2008 - Automatic Time Series Forecasting: The forecast Package for R

[4] Vrugt2026 - Artificial Intelligence and Intelligent Matter

[5] Fischer2021 - Reservoir Computing: Theory, Physical Implementations, and Applications

[6] Nakajima2020 - Physical reservoir computing -- An introductory perspective

[7] Deistler2018 - Modelle der Zeitreihenanalyse
t
[8] https://owenagnel.github.io/qreservoir/qreservoir/encoders.html#CHEEncoder, 20260412 22:32

